# Protenix Reprediction

Re-fold **target + binder** complexes (and optionally the **binder alone**) with **Protenix**, using the *exact* UmbraCore reprediction backend that the production pipeline runs — `UmbraCore.reprediction.protenix_cli`. Nothing here re-implements folding or metric normalization; the notebook just calls the pipeline functions, so pLDDT / pTM / i_pTM / pAE / i_pAE (and `i_pSAE`) come out on the same scale as a real campaign.

**Inputs**: a target PDB/CIF (one or more chains) + one or more binder sequences.
**Outputs**: a metrics table (CSV) and a predicted complex PDB per binder.

> Unlike ESMFold2, **Protenix runs in its own conda environment via a subprocess** (`UMBRA_PROTENIX_PYTHON`, normally the sibling `<env>-protenix` env). Run *this notebook* in the main `UmbraDesign` env — it writes JSONs and shells out to the Protenix env, which does the GPU work. You do **not** need torch/Protenix importable in the notebook kernel.
>
> Target **MSA/template** come from **precomputed TargetPrep data** (`TARGET_SETTINGS`), not a live ColabFold search. For a self-contained run leave `target_msa=False` (the target *template* is still built from `TARGET_PDB` coordinates).

## 1. Locate the repo and check the environment

In [1]:
import os, sys, shutil, subprocess

# Point this at the UmbraOne repo root (the folder containing `UmbraCore/`).
REPO_ROOT = os.environ.get("UMBRA_REPO_ROOT", os.path.abspath(".."))
if not os.path.isdir(os.path.join(REPO_ROOT, "UmbraCore")):
    p = os.path.abspath(os.getcwd())
    while p != os.path.dirname(p):
        if os.path.isdir(os.path.join(p, "UmbraCore")):
            REPO_ROOT = p; break
        p = os.path.dirname(p)
assert os.path.isdir(os.path.join(REPO_ROOT, "UmbraCore")), f"UmbraCore/ not found under {REPO_ROOT!r}; set UMBRA_REPO_ROOT"
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)
print("REPO_ROOT =", REPO_ROOT)

# --- Resolve the Protenix interpreter (its OWN env). Mirrors the .slurm logic:
# prefer UMBRA_PROTENIX_PYTHON, else the sibling "<env>-protenix" of this env. ---
px_python = os.environ.get("UMBRA_PROTENIX_PYTHON")
if not px_python:
    prefix = os.environ.get("CONDA_PREFIX", "")
    cand = f"{prefix}-protenix/bin/python" if prefix else ""
    if cand and os.path.exists(cand):
        px_python = cand
        os.environ["UMBRA_PROTENIX_PYTHON"] = px_python
assert px_python and os.path.exists(px_python), (
    "Protenix interpreter not found. Set UMBRA_PROTENIX_PYTHON to the python of "
    "your '<env>-protenix' environment (built by envs/install_umbra_lean.sh).")
print("UMBRA_PROTENIX_PYTHON =", px_python)

# Data root (model weights / CCD) for the Protenix env.
for k in ("PROTENIX_DATA_ROOT_DIR", "PROTENIX_ROOT_DIR"):
    print(f"{k} =", os.environ.get(k, "<unset>"))

# Quick GPU probe INSIDE the Protenix env (that's where folding actually runs).
try:
    out = subprocess.run(
        [px_python, "-c", "import torch; print(torch.__version__, torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')"],
        capture_output=True, text=True, timeout=120)
    print("Protenix env torch/CUDA:", out.stdout.strip() or out.stderr.strip())
except Exception as e:
    print("Could not probe Protenix env torch:", e)

REPO_ROOT = /scratch/nickel/UmbraOne_patched
UMBRA_PROTENIX_PYTHON = /work/lpdi/users/nickel/miniforge3/envs/UmbraDesignTest2-protenix/bin/python
PROTENIX_DATA_ROOT_DIR = /scratch/nickel/UmbraOne_patched/params/protenix
PROTENIX_ROOT_DIR = /scratch/nickel/UmbraOne_patched/params/protenix


Protenix env torch/CUDA: 2.7.1+cu126 True NVIDIA L40S


## 2. Configuration

`PROTENIX` mirrors the `protenix` block of `UmbraDesign/settings_advanced/blocks/reprediction.json` (the production reprediction defaults). Change what you need:

- **`seeds`** — comma-separated diffusion seeds (string), e.g. `"101"` or `"101,102"`.
- **`sample`** — diffusion samples per seed. Production uses `2`.
- **`model_name`** — `protenix-v2`.
- **`target_msa`** — condition the target on an MSA. Requires **precomputed** MSA via `TARGET_SETTINGS` (a TargetPrep JSON); left `False` here for a self-contained run.
- **`target_template`** — use the target structure as a template. `True` works from `TARGET_PDB` coordinates alone (no TargetPrep needed).
- **`binder_msa` / `binder_template`** — MSA / self-template for the binder. De novo binders usually leave both `False`.
- **`timeout_seconds`** — per-batch subprocess timeout (Protenix runs are long).

In [2]:
# ---- Target -------------------------------------------------------------
TARGET_PDB    = "/scratch/nickel/Reprediction_PD1_designs/Target/hPD1.pdb"     # PDB or CIF of the target (real coordinates)
TARGET_CHAINS = "A"                          # comma-separated target chain id(s), e.g. "A" or "A,B"

# ---- Binder sequences to repredict --------------------------------------
BINDERS = []
BINDERS_CSV = "/scratch/nickel/Reprediction_PD1_designs/PD1_duo_DesignSummary.csv"   # or a CSV path with columns name,seq

# ---- Output -------------------------------------------------------------
OUTPUT_DIR = os.path.abspath("./protenix_repredict_out")

# ---- Protenix reprediction settings (mirrors reprediction.json) ---------
PROTENIX = {
    "model_name": "protenix-v2",
    "seeds": "101",
    "sample": 1,
    "target_msa": True,          # True requires a precomputed MSA via TARGET_SETTINGS (TargetPrep JSON)
    "binder_msa": False,
    "target_template": True,      # built from TARGET_PDB coordinates -- no TargetPrep needed
    "binder_template": False,
    "use_framework_msa": False,   # nanobody/scFv framework-MSA cache; off for generic binders
    "framework_msa_cache_dir": None,
    "use_tfg_guidance": False,
    "predict_binder_alone": False,
    "timeout_seconds": 7200,
    "extra_args": None,
    "keep_batch_inputs": False,
    "keep_monomer_batch_inputs": False,
    "batch_accumulate": True,
    "batch_min_af2": 200,
    "batch_min_esm": 20,
}

# Optional: a TargetPrep/target settings dict (needed only for target-MSA lookup). None is fine.
TARGET_SETTINGS_PATH = "/scratch/nickel/Reprediction_PD1_designs/Target/hPD1/hPD1.json"
# hPD1.json IS the raw Protenix job template (a list containing one job dict with
# the target's precomputed MSA/template paths) -- NOT a "target settings" dict itself.
# target_settings just needs to POINT at it via "targetprep_json_path"; write_protenix_complex_json
# (Priority 2) resolves that key and hands the path to build_protenix_job_from_template,
# which loads the list-shaped JSON and injects the binder as the last protein chain.
TARGET_SETTINGS = None
if TARGET_SETTINGS_PATH is not None:
    TARGET_SETTINGS = {"targetprep_json_path": os.path.abspath(TARGET_SETTINGS_PATH)}


# Fold the binder monomer too (Binder_pLDDT / Binder_pTM / Binder_pAE)?
PREDICT_BINDER_ALONE = False

## 3. Assemble the pipeline inputs

Build the `candidates`, `advanced_settings`, and `design_paths` structures exactly as the pipeline expects them.

In [3]:
import csv, re

def load_binders():
    if BINDERS_CSV:
        rows = []
        with open(BINDERS_CSV) as fh:
            for r in csv.DictReader(fh):
                rows.append((r["name"].strip(), re.sub(r"[^A-Za-z]", "", r["sequence"]).upper()))
        return rows
    return [(n, re.sub(r"[^A-Za-z]", "", s).upper()) for n, s in BINDERS]

binders = load_binders()
assert binders, "No binder sequences provided"
candidates = [{"name": n, "seq": s} for n, s in binders]
for c in candidates:
    print(f"{c['name']:40s} len={len(c['seq'])}")

# advanced_settings just needs the backend flag + the protenix block.
advanced_settings = {"prediction_backend": "protenix", "protenix": PROTENIX}

# design_paths: the backend writes complex PDBs to MPNN/ and monomers to MPNN/Binder/.
design_paths = {
    "MPNN":        os.path.join(OUTPUT_DIR, "MPNN"),
    "MPNN/Binder": os.path.join(OUTPUT_DIR, "MPNN", "Binder"),
}
for p in design_paths.values():
    os.makedirs(p, exist_ok=True)
failure_csv = os.path.join(OUTPUT_DIR, "failures.csv")
# Pre-create failure_csv with headers -- update_failures() (called on any fold
# failure) does pd.read_csv(failure_csv), which raises FileNotFoundError on a
# fresh output dir and masks whatever the real failure was. generate_filter_pass_csv
# is a no-op if the file already exists.
from UmbraCore.general.utils import generate_filter_pass_csv
generate_filter_pass_csv(failure_csv, {})

assert os.path.isfile(TARGET_PDB), f"TARGET_PDB not found: {TARGET_PDB}"

PD1_duo_l150_s59861_mpnn1                len=150
PD1_duo_l149_s38460_mpnn10               len=149
PD1_duo_l148_s75252_mpnn19               len=148
PD1_duo_l147_s78011_mpnn6                len=147
PD1_duo_l146_s60537_mpnn2                len=146
PD1_duo_l146_s4790_mpnn8                 len=146
PD1_duo_l146_s20240_mpnn9                len=146
PD1_duo_l144_s22658_mpnn1                len=144
PD1_duo_l144_s25328_mpnn2                len=144
PD1_duo_l142_s45966_mpnn2                len=142
PD1_duo_l142_s31803_mpnn16               len=142
PD1_duo_l141_s93450_mpnn2                len=141
PD1_duo_l140_s31721_mpnn8                len=140
PD1_duo_l138_s47892_mpnn16               len=138
PD1_duo_l138_s32967_mpnn19               len=138
PD1_duo_l136_s46826_mpnn8                len=136
PD1_duo_l136_s50942_mpnn9                len=136
PD1_duo_l136_s99931_mpnn4                len=136
PD1_duo_l135_s40505_mpnn3                len=135
PD1_duo_l134_s54007_mpnn2                len=134
PD1_duo_l134_s70048_

## 4. (Optional) Target MSA via precomputed TargetPrep

Protenix does **not** search an MSA on the fly. To condition the target on an MSA, set `PROTENIX["target_msa"] = True` and point `TARGET_SETTINGS` at the **TargetPrep JSON** produced for this target (the same one the campaign used) — it carries the `pairedMsaPath` / `unpairedMsaPath` / `templatesPath` the backend re-anchors and splices in. The target *template* (`target_template=True`) is built from `TARGET_PDB` coordinates and needs none of this.

Leave `target_msa=False` (the default here) for a self-contained, template-only run.

In [4]:
if PROTENIX["target_msa"]:
    assert TARGET_SETTINGS is not None, (
        "target_msa=True needs TARGET_SETTINGS (a TargetPrep JSON dict) providing the "
        "precomputed target MSA (pairedMsaPath / unpairedMsaPath). Set target_msa=False "
        "for a template-only run, or load your TargetPrep JSON into TARGET_SETTINGS.")
    print("Target MSA conditioning ON; using TARGET_SETTINGS keys:", list(TARGET_SETTINGS)[:12])
else:
    print("target_msa is False; target folded template-only "
          f"(target_template={PROTENIX['target_template']}, from TARGET_PDB coordinates).")

Target MSA conditioning ON; using TARGET_SETTINGS keys: ['targetprep_json_path']


## 5. Repredict the target + binder complex

Calls `predict_binder_complex_protenix_batch` — the same function `UmbraCore.reprediction.pipeline` dispatches to when `prediction_backend == "protenix"`. It writes one JSON per candidate, runs a single batched `protenix predict` in the Protenix env, writes a complex PDB, and returns normalized metrics.

In [5]:
from UmbraCore.reprediction.protenix_cli import (
    predict_binder_complex_protenix_batch,
    predict_binder_alone_protenix_batch,
)

complex_results = predict_binder_complex_protenix_batch(
    candidates=candidates,
    target_pdb=TARGET_PDB,
    target_chains=TARGET_CHAINS,
    length=len(candidates[0]["seq"]),
    design_paths=design_paths,
    advanced_settings=advanced_settings,
    filters={},
    failure_csv=failure_csv,
    target_settings=TARGET_SETTINGS,
)

# complex_results: {name: (stats, ok)} where stats = {1: {pLDDT,pTM,i_pTM,pAE,i_pAE,i_pSAE}}
for name, (stats, ok) in complex_results.items():
    print(f"{name:40s} ok={ok}  {stats.get(1)}")

PD1_duo_l150_s59861_mpnn1                ok=True  {'pLDDT': 0.908, 'pTM': 0.9279, 'i_pTM': 0.9049, 'pAE': 4.19, 'i_pAE': 5.48, 'i_pSAE': None}
PD1_duo_l149_s38460_mpnn10               ok=True  {'pLDDT': 0.8949, 'pTM': 0.8378, 'i_pTM': 0.7188, 'pAE': 5.63, 'i_pAE': 8.46, 'i_pSAE': None}
PD1_duo_l148_s75252_mpnn19               ok=True  {'pLDDT': 0.8993, 'pTM': 0.9065, 'i_pTM': 0.8592, 'pAE': 4.46, 'i_pAE': 5.91, 'i_pSAE': None}
PD1_duo_l147_s78011_mpnn6                ok=True  {'pLDDT': 0.8448, 'pTM': 0.8097, 'i_pTM': 0.7143, 'pAE': 7.2, 'i_pAE': 10.11, 'i_pSAE': None}
PD1_duo_l146_s60537_mpnn2                ok=True  {'pLDDT': 0.8744, 'pTM': 0.8926, 'i_pTM': 0.9223, 'pAE': 5.09, 'i_pAE': 6.51, 'i_pSAE': None}
PD1_duo_l146_s4790_mpnn8                 ok=True  {'pLDDT': 0.8467, 'pTM': 0.763, 'i_pTM': 0.6479, 'pAE': 7.4, 'i_pAE': 11.26, 'i_pSAE': None}
PD1_duo_l146_s20240_mpnn9                ok=True  {'pLDDT': 0.8755, 'pTM': 0.909, 'i_pTM': 0.9258, 'pAE': 5.11, 'i_pAE': 6.41, 'i_pSAE': N

## 6. (Optional) Repredict the binder alone

Folds each binder as a monomer to get `Binder_pLDDT / Binder_pTM / Binder_pAE` — the monomer-confidence metrics used in filtering.

In [6]:
mono_results = {}
if PREDICT_BINDER_ALONE:
    mono_results = predict_binder_alone_protenix_batch(
        candidates=candidates,
        length=len(candidates[0]["seq"]),
        trajectory_pdb=TARGET_PDB,   # unused for folding; binder is folded from sequence
        binder_chain="B",
        prediction_models=[0],
        advanced_settings=advanced_settings,
        design_paths=design_paths,
    )
    for name, stats in mono_results.items():
        print(f"{name:40s} {stats.get(1, stats)}")

## 7. Collect results into a table

In [7]:
import pandas as pd

rows = []
for c in candidates:
    name = c["name"]
    stats, ok = complex_results.get(name, ({}, False))
    m = stats.get(1, {}) if isinstance(stats, dict) else {}
    row = {
        "name": name,
        "length": len(c["seq"]),
        "ok": ok,
        "pLDDT": m.get("pLDDT"),
        "pTM": m.get("pTM"),
        "i_pTM": m.get("i_pTM"),
        "pAE": m.get("pAE"),
        "i_pAE": m.get("i_pAE"),
        "i_pSAE": m.get("i_pSAE"),
        "complex_pdb": os.path.join(design_paths["MPNN"], f"{name}_model1.pdb"),
    }
    if PREDICT_BINDER_ALONE:
        bm = mono_results.get(name, {})
        bm = bm.get(1, bm) if isinstance(bm, dict) else {}
        row.update({
            "Binder_pLDDT": bm.get("Binder_pLDDT"),
            "Binder_pTM": bm.get("Binder_pTM"),
            "Binder_pAE": bm.get("Binder_pAE"),
        })
    rows.append(row)

df = pd.DataFrame(rows)
out_csv = os.path.join(OUTPUT_DIR, "reprediction_metrics.csv")
df.to_csv(out_csv, index=False)
print("Wrote", out_csv)
df

Wrote /scratch/nickel/Reprediction_PD1_designs/Protenix-V2/human/protenix_repredict_out/reprediction_metrics.csv


,name,length,ok,pLDDT,pTM,i_pTM,pAE,i_pAE,i_pSAE,complex_pdb
0,PD1_duo_l150_s59861_mpnn1,150,True,0.9080,0.9279,0.9049,4.19,5.48,None,/scratch/nickel/Reprediction_PD1_designs/Prote...
1,PD1_duo_l149_s38460_mpnn10,149,True,0.8949,0.8378,0.7188,5.63,8.46,None,/scratch/nickel/Reprediction_PD1_designs/Prote...
2,PD1_duo_l148_s75252_mpnn19,148,True,0.8993,0.9065,0.8592,4.46,5.91,None,/scratch/nickel/Reprediction_PD1_designs/Prote...
3,PD1_duo_l147_s78011_mpnn6,147,True,0.8448,0.8097,0.7143,7.20,10.11,None,/scratch/nickel/Reprediction_PD1_designs/Prote...
4,PD1_duo_l146_s60537_mpnn2,146,True,0.8744,0.8926,0.9223,5.09,6.51,None,/scratch/nickel/Reprediction_PD1_designs/Prote...
...,...,...,...,...,...,...,...,...,...,...
91,PD1_duo_l81_s68430_mpnn2,81,True,0.8517,0.7991,0.6223,7.65,10.73,None,/scratch/nickel/Reprediction_PD1_designs/Prote...
92,PD1_duo_l81_s13409_mpnn7,81,True,0.9241,0.9400,0.9306,3.20,3.96,None,/scratch/nickel/Reprediction_PD1_designs/Prote...
93,PD1_duo_l81_s59351_mpnn5,81,True,0.9084,0.9340,0.9315,3.71,4.59,None,/scratch/nickel/Reprediction_PD1_designs/Prote...
94,PD1_duo_l80_s50417_mpnn2,80,True,0.9209,0.9199,0.9126,3.29,4.15,None,/scratch/nickel/Reprediction_PD1_designs/Prote...


## Notes

- **Separate environment**: Protenix folds in its own conda env via subprocess. This notebook only needs `UmbraCore` importable and `UMBRA_PROTENIX_PYTHON` set; the GPU work happens there. The launcher (`protenix_launcher.py`) pre-registers a pure-PyTorch LayerNorm fallback, so runs survive clusters that can't build the fused CUDA kernel.
- **Metric scales** match the pipeline: pLDDT / pTM / i_pTM are 0–1; pAE / i_pAE are in the AF3-style 0–31.75 Å bin space. `i_pSAE` is the interface pSAE score. These are what your filter thresholds are tuned against.
- **`i_pTM` / `i_pAE` / `i_pSAE`** are the interface metrics — the ones to watch for binder quality against the target.
- **Target MSA**: precomputed only — point `TARGET_SETTINGS` at your TargetPrep JSON and set `target_msa=True`. There is no on-the-fly ColabFold search in this backend (that's the ESMFold2 notebook).
- **Target template**: `target_template=True` builds the template straight from `TARGET_PDB` coordinates — no TargetPrep needed.
- **Seeds / samples**: `seeds` (comma-separated string) × `sample` diffusion samples per seed drive compute and diversity. Production uses `seeds="101"`, `sample=2`.
- **Timeout**: `timeout_seconds` bounds each batched `protenix predict` subprocess; raise it for large batches or long targets.
- **Data root**: model weights / CCD are resolved from `PROTENIX_DATA_ROOT_DIR` / `PROTENIX_ROOT_DIR` in the Protenix env; downloaded on first use and cached.